<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/p9_type2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part4/ch9/p9_type2.ipynb)

In [12]:
## 문제 정의 : 농작물에서 농약 검출 여부를 예측하시오
## 예측 컬럼 : 농약검출여부 (0: 미검출, 1: 검출, 2: 재검사 필요)
## 제출 파일명 : result.csv
## pred : 예측된 검출 여부  1개 컬럼 포함
## 평가지표 : macro F1 Score

## 주어진 데이터 : 학습용 데이터(farm_train.csv) / 평가용 데이터 (farm_test.csv)

# 1. 라이브러리 및 데이터 ## 이번 문제는 분류임(다중)
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_test.csv")
# 데이터 샘플 확인
# print(train.head(4), "\n", test.head(4))

# 2. 탐색적 데이터 분석 (크기 / 자료형 / 결측치 / 빈도 수)
# print(train.shape, test.shape) # (4000, 9) (1000, 8)
# print(train.info(), test.info()) # int 1 / float 3 / object 4 (train int 1 추가) : 수치 + 범주 혼합 데이터
# print(train.isnull().sum(), "\n", test.isnull().sum()) # 결측치 없음 - 기출에는 결측치 없이 나오는 듯
# print(train['농약검출여부'].value_counts()) # 2    1989 / 0    1758 / 1     253 (10:8:1 정도?)

# 3. 데이터 전처리 (원-핫 인코딩 포함)
target = train.pop('농약검출여부') # 타겟 분리
train = pd.get_dummies(train)
test = pd.get_dummies(test)
print(train.shape, test.shape) # 인코딩 후 데이터 크기 확인 : (4000, 28) (1000, 28)

# 4. 검증 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 5. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_train, y_train)
pred = rf.predict(X_val)

## 평가 : f1-score 'macro'
from sklearn.metrics import f1_score
f1 = f1_score(y_val, pred, average='macro')
print("RF f1 : ", f1) # 결과 : RF f1 :  0.8532014300116062

## 5-1. LightGBM
import lightgbm as lgb
lg = lgb.LGBMClassifier(random_state=0, verbose=-1)
lg.fit(X_train, y_train)
pred = lg.predict(X_val)

f1_lg = f1_score(y_val, pred, average='macro')
print("LGB f1 : ", f1_lg) # 결과 : LGB f1 :  0.9100316620356779
## LGBM 성능이 우수

# 6. 예측 수행 및 결과 파일 생성
pred = lg.predict(test)
result = pd.DataFrame({'pred': pred})
result.to_csv("result.csv", index=False)

## 검증
print(pred.shape) ## 처음 test 데이터 크기가 1000행 이므로 제출 데이터 크기도 1000 이어야 함 : (1000,) 확인 완료
file = pd.read_csv("result.csv")
file # 확인

(4000, 28) (1000, 28)
RF f1 :  0.8532014300116062
LGB f1 :  0.9100316620356779
(1000,)


,pred
0,2
1,0
2,0
3,2
4,0
...,...
995,2
996,2
997,2
998,2


In [ ]:
# 문제정의
# 평가: f1-macro
# target: 농약검출여부
# 최종파일: result.csv(컬럼 1개 pred)

# 라이브러리 및 데이터 불러오기
import pandas as pd
train = pd.read_csv("farm_train.csv")
test = pd.read_csv("farm_test.csv")
# train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_train.csv")
# test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_test.csv")


# 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\n ===== 데이터 샘플 =====")
print(train.head())

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train object컬럼 unique 수 =====")
print(train.describe(include='object'))

print("\n ===== test object컬럼 unique 수 =====")
print(test.describe(include='object'))

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum().sum())

print("\n ===== target unique 수 =====")
print(train['농약검출여부'].value_counts())

===== 데이터 크기 =====
Train Shape: (4000, 9)
Test Shape: (1000, 8)

 ===== 데이터 샘플 =====
           농업면적    연도  지역       비료사용량        비료잔여량 작물종류 토양유형  농약검출여부 등급
0  20079.652837  2004  대구  407.985516   146.290507   보리   점토       2  C
1  73858.643204  2012  울산  221.229692  1967.333638    밀   점토       0  B
2  65718.150861  2012  강원  370.967205  2253.522610    쌀   점토       0  B
3  37366.182902  2005  광주  274.128236  1487.535265    쌀   양토       0  B
4  81515.151289  2007  충남  213.410655   683.306745    쌀   양토       1  B

 ===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   농업면적    4000 non-null   float64
 1   연도      4000 non-null   int64  
 2   지역      4000 non-null   object 
 3   비료사용량   4000 non-null   float64
 4   비료잔여량   4000 non-null   float64
 5   작물종류    4000 non-null   object 
 6   토양유형    4000 non-null   object 
 7   농약검출여부  4000 

In [ ]:
# 데이터 다시 불러오기
train = pd.read_csv("farm_train.csv")
test = pd.read_csv("farm_test.csv")
# train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_train.csv")
# test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/farm_test.csv")

# 데이터 전처리
target = train.pop('농약검출여부')

# 원핫 인코딩
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 검증데이터 분리
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 랜덤포레스트
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

# macro F1 score
from sklearn.metrics import f1_score
print("랜덤포레스트")
print(f1_score(y_val, pred, average='macro'))

# LightGBM
import lightgbm as lgb
lg = lgb.LGBMClassifier(random_state=0, verbose=-1)
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)
print("lightgbm")
print(f1_score(y_val, pred, average='macro'))

# 최종 제출 파일 (lightGBM)
pred = lg.predict(test)
result = pd.DataFrame({
    'pred':pred
})
result.to_csv("result.csv", index=False)


랜덤포레스트
0.8532014300116062
lightgbm
0.9100316620356779


In [ ]:
# 1. pred 행의 수와 test의 행의 수 비교
print("pred: ",pred.shape) # test 행의 수: 1000

# 2. 생성한 csv 확인
print(pd.read_csv("result.csv").head())

pred:  (1000,)
   pred
0     2
1     0
2     0
3     2
4     0
